In [ ]:
!pip install enoslib ipywidgets --break-system-packages

In [ ]:
!ssh rennes.grid5000.fr hostname

In [ ]:
!ssh-add ~/.ssh/id_ed25519

### G5K connection

Setup the `.python-grid5000.yaml` file with the username and password used to login to grid5000.

The file should look like this:
```yaml
username: G5K_LOGIN
password: G5K_password
```

-> required in order for the enoslib calls to g5k to work.

In addition to this, make sure to add these lines to your ssh configuration:

```text
Host g5k
    User G5K_LOGIN
    HostName access.grid5000.fr
    ForwardAgent no

Host !access.grid5000.fr *.grid5000.fr
    User G5K_LOGIN
    ProxyJump G5K_LOGIN@access.grid5000.fr
    StrictHostKeyChecking no
    UserKnownHostsFile /dev/null
    ForwardAgent yes

Host access.grid5000.fr
    User G5K_LOGIN
    StrictHostKeyChecking no
    UserKnownHostsFile /dev/null
    ForwardAgent yes

Host *.g5k
    User G5K_LOGIN
    ProxyCommand ssh g5k -W "$(basename %h .g5k):%p"
    ForwardAgent no
```

Make sure to replace `G5K_LOGIN` with your g5k username (the one from the site)

In [1]:
from grid5000 import Grid5000
import enoslib as en
import logging
import os
from datetime import datetime, timedelta

conf_file = os.path.join(os.environ.get("HOME"), ".python-grid5000.yaml")  # type: ignore
gk = Grid5000.from_yaml(conf_file)

# map each cluster to its site
cluster_to_site = {}
for site in gk.sites.list():
    for cluster in site.clusters.list():
        cluster_to_site[cluster.uid] = site.uid

# JOB CONFIGURATION
JOB_NAME = "fcquic_local_eval"
# CLUSTER="vianden"
CLUSTER = "larochette"
JOB_WALLTIME = timedelta(hours=2, minutes=0)

# usage policy check: the job cannot cross the day to night boundary at 7pm if it was submitted before 5 pm. Any job started after 5 pm can cross the boundary
# I had the issue once so this check is there to avoid receiving a usage policy violation email...
datetime_now = datetime.now()
job_end_dt = datetime_now + JOB_WALLTIME
if (
    datetime_now.hour <= 17 and job_end_dt.hour >= 19 and datetime_now.weekday() <= 5
):  # only check during weekdays
    raise RuntimeError(
        "This job reservation will violate the usage policy and will cross the day night boundary"
    )

# Display some general information about the library
en.check()
# Enable rich logging
_ = en.init_logging()

conf = en.G5kConf.from_settings(
    job_name=JOB_NAME,
    walltime=str(JOB_WALLTIME),
    env_name="debian13-nfs",  # using debian13 here, was 12 before
    job_type=["deploy"],
).add_machine(
    roles=["server"],
    cluster=CLUSTER,
    nodes=1,
)

# This will validate the configuration, but not reserve resources yet
provider = en.G5k(conf)

[WARNING]: failed to patch stdout/stderr for fork-safety: 'OutStream' object
has no attribute 'buffer'
[WARNING]: failed to reconfigure stdout/stderr with custom encoding error
handler: 'OutStream' object has no attribute 'reconfigure'


_____        ___  ____  _ _ _
 | ____|_ __  / _ \/ ___|| (_) |__
 |  _| | '_ \| | | \___ \| | | '_ \
 | |___| | | | |_| |___) | | | |_) |
 |_____|_| |_|\___/|____/|_|_|_.__/  10.6.0

 • Documentation: ]8;id=500362;https://discovery.gitlabpages.inria.fr/enoslib/\https://discovery.gitlabpages.inria.fr/enoslib/]8;;\                            
 • Source: ]8;id=114154;https://gitlab.inria.fr/discovery/enoslib\https://gitlab.inria.fr/discovery/enoslib]8;;\                                         
 • Chat: ]8;id=12392;https://framateam.org/enoslib\https://framateam.org/enoslib]8;;\

                         Dependency check                         
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Provider      ┃    Status     ┃ Hint                           ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Chameleon     │ NOT INSTALLED │ pip install enoslib[chameleon] │
│ ChameleonKVM  │ NOT INSTALLED │ pip install enoslib[chameleon] │
│ ChameleonEdge │ NOT INSTALLED │ pip install enoslib[chameleon] │
│ Fabric        │ NOT INSTALLED │ pip install enoslib[fabric]    │
│ Distem        │ NOT INSTALLED │ pip install enoslib[distem]    │
│ IOT-lab       │ NOT INSTALLED │ pip install enoslib[iotlab]    │
│ Grid'5000     │   INSTALLED   │                                │
│ Openstack     │ NOT INSTALLED │ pip install enoslib[chameleon] │
│ Vagrant       │ NOT INSTALLED │ pip install enoslib[vagrant]   │
│ VMonG5k       │   INSTALLED   │                                │
└───────────────┴───────────────┴────────────────────────────────┘

                                Connectivity check                                 
┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Provider  ┃ Key                 ┃ Connectivity ┃ Hint                           ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Grid'5000 │ ssh:access          │      ✅      │ Connection to access.grid5000… │
│ Grid'5000 │ ssh:access:frontend │      ✅      │ Connection Host(rennes.grid50… │
│ Grid'5000 │ api:access          │      ✅      │                                │
│ VMonG5k   │ access              │      ❔      │ Check G5k status               │
└───────────┴─────────────────────┴──────────────┴────────────────────────────────┘

In [ ]:
print("Reserving resources...")

# Get actual resources
roles, networks = provider.init()
display(roles)
display(networks)

# Fill in network information from nodes
roles = en.sync_info(roles, networks)

with en.actions(roles=roles, gather_facts=True) as a:
    a.apt(
        task_name="Install packages",
        name=[
            "tcpdump",
            "cmake",
            "clang",
            "python3.13-venv",
            "python-is-python3",
            "python3-pip",
            "btop",
            "htop",
        ],
        state="present",
    )

    # install frr
    a.file(
        task_name="Ensure apt keyring directory exists",
        path="/usr/share/keyrings",
        state="directory",
        mode="0755",
    )
    a.get_url(
        task_name="Download FRR GPG key",
        url="https://deb.frrouting.org/frr/keys.gpg",
        dest="/usr/share/keyrings/frrouting.gpg",
        mode="0644",
    )
    a.apt_repository(
        task_name="Add FRR apt repository",
        repo="deb [signed-by=/usr/share/keyrings/frrouting.gpg] https://deb.frrouting.org/frr {{ ansible_distribution_release }} frr-stable",
        filename="frr",
        state="present",
    )
    a.apt(
        task_name="Install FRR packages",
        name=["frr", "frr-pythontools"],
        state="present",
        update_cache=True,
    )

    # rust
    a.get_url(
        task_name="Download rustup installer",
        url="https://sh.rustup.rs",
        dest="/tmp/rustup-init.sh",
        mode="0755",
    )
    a.shell(
        task_name="Install Rust stable (rustup)",
        cmd="sh /tmp/rustup-init.sh -y --default-toolchain stable",
        creates="/root/.cargo/bin/rustup",
    )
    a.shell(
        task_name="Install Rust nightly toolchain",
        cmd="/root/.cargo/bin/rustup toolchain install nightly",
    )

    results = a.results

Reserving resources...


INFO     [G5k] Submitting {'name': 'fcquic_local_eval', 'types':         ]8;id=794786;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=306301;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#304\304]8;;\
         ['deploy', 'origin=enoslib_g5k'], 'resources':                                      
         "{cluster='larochette'}/nodes=1,walltime=2:00:00", 'command':                       
         'sleep 31536000', 'queue': 'default'} on luxembourg                                 

INFO     [G5k] Waiting for 5 seconds before next OAR job(s) check...     ]8;id=604726;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=198854;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#337\337]8;;\

INFO     [G5k] Job 280387 on luxembourg: no schedule estimate            ]8;id=281022;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=806747;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#352\352]8;;\

INFO     [G5k] Waiting for 10 seconds before next OAR job(s) check...    ]8;id=558654;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=517147;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#337\337]8;;\

...

INFO     [G5k] Deploying ['larochette-4.luxembourg.grid5000.fr'] on     ]8;id=451047;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=433097;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#1102\1102]8;;\
         luxembourg                                                                          

INFO     [G5k] Preparing deployment on luxembourg with config:          ]8;id=30005;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=92759;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#1104\1104]8;;\
         {'environment': 'debian13-nfs', 'key': 'ssh-ed25519 AAAAC3NzaC                      
         1lZDI1NTE5AAAAIHcvopjcrP1u/Uk26PdY8dPbs2Y8x8fyO9Rcu6e0+71F                          
         corentin.detry@student.uclouvain.be\n', 'nodes':                                    
         ['larochette-4.luxembourg.grid5000.fr']}                                            

INFO     [G5k] Waiting for the end of deployment                        ]8;id=605028;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=44082;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#1117\1117]8;;\
         [D-4894b5e7-38d3-4da9-9ce2-931249600cae](processing on                              
         luxembourg)                                                                         

INFO     [G5k] Waiting for the end of deployment                        ]8;id=950658;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=80719;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#1117\1117]8;;\
         [D-4894b5e7-38d3-4da9-9ce2-931249600cae](processing on                              
         luxembourg)                                                                         

INFO     [G5k] Waiting for the end of deployment                        ]8;id=640099;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=890712;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#1117\1117]8;;\
         [D-4894b5e7-38d3-4da9-9ce2-931249600cae](processing on                              
         luxembourg)                                                                         

In [ ]:
# print os and kernel versions
from enoslib.api import Results

res: Results = en.run_command("uname -a", roles=roles)
print([res.stdout for res in res])

### Uploading project with SCP 

In [ ]:
import subprocess

local_bin_dir = f"/home/corentin/fcquic_applications_master_thesis"
remote_bin_dir = "/tmp/chat"

for role, nodes in roles.items():
    for node in nodes:
        host = node.address
        print(f"pushing binary to {host} (role: {role})")

        subprocess.run(
            ["ssh", host, f"mkdir -p {remote_bin_dir}/fcquic_chat"], check=True
        )
        subprocess.run(
            ["ssh", host, f"mkdir -p {remote_bin_dir}/evaluations"], check=True
        )
        subprocess.run(
            ["ssh", host, f"mkdir -p {remote_bin_dir}/multicast-quic"], check=True
        )

        subprocess.run(
            [
                "rsync",
                "-az",
                "--exclude=logs/",
                "--exclude=baseline_logs/",
                "--exclude=tcp_logs/",
                "--exclude=tquic_logs/",
                "--exclude=target/",
                f"{local_bin_dir}/fcquic_chat/",
                f"{host}:{remote_bin_dir}/fcquic_chat/",
            ],
            check=True,
        )
        subprocess.run(
            [
                "rsync",
                "-az",
                "--exclude=target/",
                f"{local_bin_dir}/multicast-quic/",
                f"{host}:{remote_bin_dir}/multicast-quic/",
            ],
            check=True,
        )
        subprocess.run(
            [
                "rsync",
                "-az",
                "--exclude=grid5000/",
                "--exclude=venv/",
                "--exclude=results/",
                f"{local_bin_dir}/evaluations/",
                f"{host}:{remote_bin_dir}/evaluations/",
            ],
            check=True,
        )

print("pushed project to node")

#### Rsync script.npf

In [ ]:
import subprocess

for role, nodes in roles.items():
    for node in nodes:
        host = node.address

        subprocess.run(
            [
                "rsync",
                "-az",
                f"{local_bin_dir}/evaluations/tests/script.npf",
                f"{host}:{remote_bin_dir}/evaluations/tests/script.npf",
            ],
            check=True,
        )

        subprocess.run(
            [
                "rsync",
                "-az",
                f"{local_bin_dir}/evaluations/run_test.sh",
                f"{host}:{remote_bin_dir}/evaluations/run_test.sh",
            ],
            check=True,
        )

### Running the experiment

In [ ]:
en.run_command(
    f"pip install graphviz networkx pandas brokenaxes --break-system-packages",
    roles=roles,
)

*Important:* SSH into the machine `root@vianden-1.luxembourg.grid5000.fr`, then run the following commands:
- `cd /tmp/chat/evaluations`
- `python -m venv venv`
- `source ./venv/bin/activate`
- `pip install npf`

In [ ]:
en.run_command(f"cd {remote_bin_dir}/fcquic_chat && cargo build --release", roles=roles)

In [ ]:
# TEST_DIR_NAME = "receivers"
# TOPO_CONF_NAME = "receivers"
TEST_DIR_NAME = "data"
TOPO_CONF_NAME = "data"
USE_POISSON = "true"

In [ ]:
for role, nodes in roles.items():
    for node in nodes:
        host = node.address
        subprocess.run(
            [
                "ssh",
                "root@" + host,
                f"cd {remote_bin_dir}/evaluations/ && bash run_test.sh {TEST_DIR_NAME} {TOPO_CONF_NAME} {USE_POISSON}",
            ],
            check=True,
        )

### Collecting data

In [ ]:
import subprocess
import os

poisson_str = "poisson" if USE_POISSON else "uniform"
result_filename = f"npf_out_{TOPO_CONF_NAME}_{poisson_str}"

remote_out_dir = f"{remote_bin_dir}/evaluations/tests/{TEST_DIR_NAME}/out/"
local_out_dir = f"{local_bin_dir}/evaluations/tests/{TEST_DIR_NAME}/out/"

os.makedirs(local_out_dir, exist_ok=True)

for role, nodes in roles.items():
    for node in nodes:
        host = node.address
        print(f"downloading results from {host}")
        subprocess.run(
            [
                "rsync",
                "-az",
                f"root@{host}:{remote_out_dir}{result_filename}.csv",
                f"root@{host}:{remote_out_dir}{result_filename}-TLOAD.csv",
                local_out_dir,
            ],
            check=True,
        )

print(f"results saved to {local_out_dir}{result_filename}")

In [ ]:
import subprocess

poisson_str = "poisson" if USE_POISSON else "uniform"
result_filename = f"npf_out_{TOPO_CONF_NAME}_{poisson_str}"
graphs_dir = f"{local_bin_dir}/evaluations/graphs"
csv_path = f"{local_bin_dir}/evaluations/tests/{TEST_DIR_NAME}/out/{result_filename}.csv"
output_dir = f"{graphs_dir}/{TEST_DIR_NAME}"

os.makedirs(output_dir, exist_ok=True)

subprocess.run(
    ["python", f"{graphs_dir}/{TEST_DIR_NAME}.py", csv_path, output_dir],
    check=True,
)

print(f"graphs written to {output_dir}")

In [ ]:
# compress all related files in one tarball
subprocess.run(
    [
        "tar",
        "czf",
        f"{local_out_dir}results.tar.gz",
        f"{local_out_dir}{result_filename}.csv",
        f"{local_out_dir}{result_filename}-TLOAD.csv",
    ],
    check=True,
)

# move archive to the graph dir of the test
subprocess.run(
    [
        "mv",
        f"{local_out_dir}results.tar.gz",
        output_dir,
    ],
    check=True,
)

# delete the csvs and directories
subprocess.run(
    [
        "rm",
        f"{local_out_dir}{result_filename}.csv",
        f"{local_out_dir}{result_filename}-TLOAD.csv",
    ],
    check=True,
)

In [ ]:
# extract the tarball back into local_out_dir (inverse of the cell above)
subprocess.run(
    [
        "cp",
        f"{output_dir}/results.tar.gz",
        local_out_dir,
    ],
    check=True,
)

subprocess.run(
    [
        "tar",
        "xzf",
        f"{local_out_dir}results.tar.gz",
        "-C",
        "/",
    ],
    check=True,
)

subprocess.run(
    [
        "rm",
        f"{local_out_dir}results.tar.gz",
    ],
    check=True,
)

### Stopping the current booking

In [ ]:
provider.destroy()